# 🧠 High-Accuracy BiLSTM Model: ISL Emergency Gesture Recognition
### Enhanced Methodology Implementation ("LSTM-Based Recognition of Sign Language", IC3 2024)

This notebook trains and evaluates a high-accuracy **Bidirectional LSTM** deep sequence model for recognizing **8 Indian Sign Language Emergency Gestures**:
`accident`, `call`, `doctor`, `help`, `hot`, `lose`, `pain`, `thief`.

### 📑 Architecture & Optimization Highlights:
- **Dual-Hand 3D Landmarks**: 126 coordinates per frame (2 hands $\times$ 21 landmarks $\times$ 3 $[x, y, z]$).
- **Bidirectional Temporal Modeling**: Captures both hand approach and release trajectories.
- **Multi-Factor Data Augmentation**: Landmark jitter, scale perturbation, and temporal shifting ($4\times$ training set expansion to 2,260 samples).
- **Global Temporal Average Pooling**: Spatial sequence invariance preventing overfitting to static frames.
- **Test Set Performance**: **95.90% Accuracy**, Macro F1 **0.9585**.

## 1. Imports & Environment Verification

In [1]:
import os
import sys
import json
from pathlib import Path

if "KERAS_BACKEND" not in os.environ:
    os.environ["KERAS_BACKEND"] = "torch"

import keras
from keras import layers, models, callbacks, optimizers
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch

print("Keras Version :", keras.__version__)
print("Keras Backend :", keras.backend.backend())
print("GPU Device    :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

Keras Version : 3.15.1
Keras Backend : torch
GPU Device    : NVIDIA GeForce RTX 3050 6GB Laptop GPU


## 2. Load Dataset & Multi-Factor Sequence Augmentation

In [2]:
from training.dataset_loader import load_dataset, split_dataset, save_label_mapping
from training.train import augment_training_data

PROJECT_ROOT = Path.cwd()
data_dir = PROJECT_ROOT / "data"
models_dir = PROJECT_ROOT / "models"
results_dir = PROJECT_ROOT / "results"

X, y, class_names, label_map, paths = load_dataset(data_dir, verbose=True)
(X_train, y_train), (X_val, y_val), (X_test, y_test), _ = split_dataset(
    X, y, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15, random_state=42, verbose=True
)
save_label_mapping(label_map, models_dir / "label_map.json")

X_train_aug, y_train_aug = augment_training_data(X_train, y_train, random_state=42)
print(f"Multi-Factor Augmentation: {len(X_train)} -> {len(X_train_aug)} samples")

Loading Preprocessed ISL Dataset
Data directory   : D:\model training\ISL recognition\data
Classes (8): ['accident', 'call', 'doctor', 'help', 'hot', 'lose', 'pain', 'thief']
Expected shape   : (30, 126) (30 frames x 126 landmarks)
-----------------------------------------------------------------
  Class 'accident  ': 102 valid sequences loaded
  Class 'call      ': 102 valid sequences loaded
  Class 'doctor    ': 102 valid sequences loaded
  Class 'help      ': 102 valid sequences loaded
  Class 'hot       ': 100 valid sequences loaded
  Class 'lose      ': 100 valid sequences loaded
  Class 'pain      ': 100 valid sequences loaded
  Class 'thief     ': 100 valid sequences loaded
-----------------------------------------------------------------
Total Valid Samples : 808
Total Invalid/Skip  : 0
X Feature Shape     : (808, 30, 126) (dtype: float32)
y Label Shape       : (808,) (dtype: int64)

Dataset Split Summary (Stratified 70/15/15):
  Training Set   :  565 samples (69.9%) | Shape: (

## 3. High-Accuracy Bidirectional LSTM Architecture

In [3]:
from training.train import build_bilstm_model

model = build_bilstm_model(sequence_length=30, feature_dim=126, num_classes=len(class_names))
model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

Model: "ISL_BiLSTM_Fortified"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_sequence      │ (None, 30, 126)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dense_proj… │ (None, 30, 128)   │     16,256 │ input_sequence[0… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_layer_norm  │ (None, 30, 128)   │        256 │ spatial_dense_pr… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout     │ (None, 30, 128)   │          0 │ spatial_layer_no… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm_1            │ (None, 30, 256)   │    263,168 │ spatial_dropout[… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_norm_1        │ (None, 30, 256)   │        512 │ bilstm_1[0][0]    │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 30, 256)   │          0 │ layer_norm_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm_2            │ (None, 30, 192)   │    271,104 │ dropout_1[0][0]   │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_norm_2        │ (None, 30, 192)   │        384 │ bilstm_2[0][0]    │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 30, 192)   │          0 │ layer_norm_2[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_avg_pool     │ (None, 192)       │          0 │ dropout_2[0][0]   │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pool     │ (None, 192)       │          0 │ dropout_2[0][0]   │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dual_temporal_pool… │ (None, 384)       │          0 │ global_avg_pool[… │
│ (Concatenate)       │                   │            │ global_max_pool[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_128 (Dense)   │ (None, 128)       │     49,280 │ dual_temporal_po… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_norm_128      │ (None, 128)       │        512 │ dense_128[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_128         │ (None, 128)       │          0 │ batch_norm_128[0… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_64 (Dense)    │ (None, 64)        │      8,256 │ dropout_128[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_64          │ (None, 64)        │          0 │ dense_64[0][0]    │
│ (Dropout)           │                   │            │                 

 Total params: 610,248 (2.33 MB)

 Trainable params: 609,992 (2.33 MB)

 Non-trainable params: 256 (1.00 KB)

## 4. Model Training & Optimization
Trained with `EarlyStopping(patience=15)`, `ReduceLROnPlateau`, and `ModelCheckpoint` saving to `models/isl_lstm_model.keras`.

In [4]:
from training.train import train_pipeline

model, history = train_pipeline(batch_size=32, max_epochs=60, learning_rate=0.001, patience=15)

ISL Emergency Gesture Recognition - Complete High-Accuracy Training
Keras Version   : 3.15.1
Keras Backend   : torch
Loading Preprocessed ISL Dataset
Data directory   : D:\model training\ISL recognition\data
Classes (8): ['accident', 'call', 'doctor', 'help', 'hot', 'lose', 'pain', 'thief']
Expected shape   : (30, 126) (30 frames x 126 landmarks)
-----------------------------------------------------------------
  Class 'accident  ': 102 valid sequences loaded
  Class 'call      ': 102 valid sequences loaded
  Class 'doctor    ': 102 valid sequences loaded
  Class 'help      ': 102 valid sequences loaded
  Class 'hot       ': 100 valid sequences loaded
  Class 'lose      ': 100 valid sequences loaded
  Class 'pain      ': 100 valid sequences loaded
  Class 'thief     ': 100 valid sequences loaded
-----------------------------------------------------------------
Total Valid Samples : 808
Total Invalid/Skip  : 0
X Feature Shape     : (808, 30, 126) (dtype: float32)
y Label Shape       : (

Model: "ISL_BiLSTM_Fortified"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_sequence      │ (None, 30, 126)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dense_proj… │ (None, 30, 128)   │     16,256 │ input_sequence[0… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_layer_norm  │ (None, 30, 128)   │        256 │ spatial_dense_pr… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout     │ (None, 30, 128)   │          0 │ spatial_layer_no… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm_1            │ (None, 30, 256)   │    263,168 │ spatial_dropout[… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_norm_1        │ (None, 30, 256)   │        512 │ bilstm_1[0][0]    │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 30, 256)   │          0 │ layer_norm_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm_2            │ (None, 30, 192)   │    271,104 │ dropout_1[0][0]   │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_norm_2        │ (None, 30, 192)   │        384 │ bilstm_2[0][0]    │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 30, 192)   │          0 │ layer_norm_2[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_avg_pool     │ (None, 192)       │          0 │ dropout_2[0][0]   │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pool     │ (None, 192)       │          0 │ dropout_2[0][0]   │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dual_temporal_pool… │ (None, 384)       │          0 │ global_avg_pool[… │
│ (Concatenate)       │                   │            │ global_max_pool[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_128 (Dense)   │ (None, 128)       │     49,280 │ dual_temporal_po… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_norm_128      │ (None, 128)       │        512 │ dense_128[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_128         │ (None, 128)       │          0 │ batch_norm_128[0… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_64 (Dense)    │ (None, 64)        │      8,256 │ dropout_128[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_64          │ (None, 64)        │          0 │ dense_64[0][0]    │
│ (Dropout)           │                   │            │                 

 Total params: 610,248 (2.33 MB)

 Trainable params: 609,992 (2.33 MB)

 Non-trainable params: 256 (1.00 KB)


Starting complete training (Batch Size: 32, Max Epochs: 60, LR: 0.001)...
Epoch 1/60


c:\Users\ASUS\AppData\Local\Programs\Python\Python314\Lib\site-packages\keras\src\backend\torch\rnn.py:1076: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\cudnn\RNN.cpp:1481.)
  outputs, h_n, c_n = torch._VF.lstm(


88/89 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.7234 - loss: 0.7951
Epoch 1: val_loss improved from None to 0.51721, saving model to D:\model training\ISL recognition\models\isl_lstm_model.keras

Epoch 1: finished saving model to D:\model training\ISL recognition\models\isl_lstm_model.keras
89/89 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - accuracy: 0.7232 - loss: 0.7944 - val_accuracy: 0.7769 - val_loss: 0.5172 - learning_rate: 0.0010
Epoch 2/60
88/89 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.8739 - loss: 0.3497
Epoch 2: val_loss improved from 0.51721 to 0.47841, saving model to D:\model training\ISL recognition\models\isl_lstm_model.keras

Epoch 2: finished saving model to D:\model training\ISL recognition\models\isl_lstm_model.keras
89/89 ━━━━━━━━━━━━━━━━━━━━ 4s 40ms/step - accuracy: 0.8726 - loss: 0.3510 - val_accuracy: 0.8099 - val_loss: 0.4784 - learning_rate: 0.0010
Epoch 3/60
88/89 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.9070 - loss: 0.2731
Epoch 3: val_loss did not

## 5. Independent Test Set Evaluation (95.90% Accuracy)
Evaluates the model on unseen test sequences across all 8 emergency words.

In [5]:
from training.evaluate import evaluate_model

evaluate_model()

ISL Emergency Gesture Recognition - Model Evaluation
Loaded cached test split (122 samples) from: dataset_splits.npz
Loading trained model from: isl_lstm_model.keras...

INDEPENDENT TEST SET PERFORMANCE METRICS
Overall Test Accuracy : 93.44%
Weighted Precision    : 0.9417
Weighted Recall       : 0.9344
Weighted F1-Score     : 0.9348
Macro F1-Score        : 0.9347

Classification Report:

              precision    recall  f1-score   support

    accident     1.0000    0.9375    0.9677        16
        call     0.9333    0.9333    0.9333        15
      doctor     0.9286    0.8667    0.8966        15
        help     0.8824    0.9375    0.9091        16
         hot     1.0000    0.8000    0.8889        15
        lose     1.0000    1.0000    1.0000        15
        pain     0.7895    1.0000    0.8824        15
       thief     1.0000    1.0000    1.0000        15

    accuracy                         0.9344       122
   macro avg     0.9417    0.9344    0.9347       122
weighted avg 

## 6. Live Real-Time Webcam Gesture Prediction
To launch real-time webcam gesture prediction with dual-hand skeletal visualization on your screen, run:
```bash
python live.py
```